# Weighted least squares

If some rows are known to be noisier than others, ordinary least squares is no longer the best linear unbiased estimator. Weighted least squares gives a quiet row more influence by multiplying its squared residual by a larger weight.


## The weighted normal equations

Minimize

$$
S(b_0, b_1) = \sum_{i=1}^{n} w_i (y_i - b_0 - b_1 x_i)^2, \qquad w_i > 0.
$$

Differentiate and set the derivatives to zero:

$$
\begin{aligned}
b_0\sum w_i + b_1\sum w_i x_i &= \sum w_i y_i, \\
b_0\sum w_i x_i + b_1\sum w_i x_i^2 &= \sum w_i x_i y_i.
\end{aligned}
$$

In matrix form, with $W = \operatorname{diag}(w_i)$,

$$
X^{\mathsf T}W X\,\beta = X^{\mathsf T}W y.
$$

This is ordinary least squares on the transformed problem in which each row of $X$ and of $y$ is multiplied by $\sqrt{w_i}$. The weight belongs to the precision. If row $i$ has variance $\sigma^2/w_i$, these weights restore the Gauss–Markov condition in the transformed model.


## A three-point weighted fit

Take $(x, y) = (0, 1),\ (1, 1),\ (2, 4)$ and weights $1, 1, 4$. The last row is four times as precise as either of the others. The weighted sums are

$$
\begin{aligned}
\sum w &= 6, & \sum wx &= 9, & \sum wy &= 18, \\
\sum wx^2 &= 17, & \sum wxy &= 33.
\end{aligned}
$$

The system is

$$
\begin{aligned}
6b_0 + 9b_1 &= 18, \\
9b_0 + 17b_1 &= 33.
\end{aligned}
$$

Divide the first equation by $3$: $2b_0 + 3b_1 = 6$, so $b_0 = (6 - 3b_1)/2$. Substitute into the second equation:

$$
\begin{aligned}
9\cdot\dfrac{6 - 3b_1}{2} + 17b_1 &= 33, \\
27 + \dfrac{7}{2}b_1 &= 33, \\
b_1 &= \dfrac{12}{7}, \qquad b_0 = \dfrac{3}{7}.
\end{aligned}
$$

The predictions and residuals are

$$
\widehat y = \dfrac{3}{7},\ \dfrac{15}{7},\ \dfrac{27}{7}, \qquad
e = \dfrac{4}{7},\ -\dfrac{8}{7},\ \dfrac{1}{7}.
$$

The weighted residuals are orthogonal to the columns of $X$:

$$
\sum w_i e_i = \dfrac{4}{7} - \dfrac{8}{7} + 4\cdot\dfrac{1}{7} = 0, \qquad
\sum w_i x_i e_i = 0 - \dfrac{8}{7} + 8\cdot\dfrac{1}{7} = 0.
$$

The weighted cost is

$$
\sum w_i e_i^2 = \dfrac{16}{49} + \dfrac{64}{49} + 4\cdot\dfrac{1}{49} = \dfrac{84}{49} = \dfrac{12}{7}.
$$

Equal weights $1, 1, 1$ on the same points give a different line. The weighted fit has leaned toward the precise point $(2, 4)$.


In [1]:
# Weights 1, 1, 4 produce intercept 3/7 and slope 12/7.
from fractions import Fraction
import numpy as np
x = [0, 1, 2]
y = [1, 1, 4]
w = [1, 1, 4]
sw = sum(w)
swx = sum(wi * xi for wi, xi in zip(w, x))
swy = sum(wi * yi for wi, yi in zip(w, y))
swx2 = sum(wi * xi ** 2 for wi, xi in zip(w, x))
swxy = sum(wi * xi * yi for wi, xi, yi in zip(w, x, y))
# Solve the 2 by 2 system with Fractions through NumPy's object inverse.
gram = np.array([[Fraction(sw), Fraction(swx)], [Fraction(swx), Fraction(swx2)]])
right = np.array([Fraction(swy), Fraction(swxy)])
beta = np.linalg.inv(gram.astype(float)) @ right.astype(float)
hand = np.array([3 / 7, 12 / 7])
errors = [Fraction(yi) - (Fraction(3, 7) + Fraction(12, 7) * xi) for xi, yi in zip(x, y)]
weighted_sum = sum(wi * ei for wi, ei in zip(w, errors))
weighted_x = sum(wi * xi * ei for wi, xi, ei in zip(w, x, errors))
cost = sum(wi * ei ** 2 for wi, ei in zip(w, errors))
print("beta", beta)
print("residuals", errors)
print("weighted cost", cost)
assert np.allclose(beta, hand)
assert errors == [Fraction(4, 7), Fraction(-8, 7), Fraction(1, 7)]
assert weighted_sum == 0 and weighted_x == 0
assert cost == Fraction(12, 7)


beta [0.42857143 1.71428571]
residuals [Fraction(4, 7), Fraction(-8, 7), Fraction(1, 7)]
weighted cost 12/7


## The same points with equal weights

With $w_i = 1$, the ordinary sums are $\sum x = 3$, $\sum y = 6$, $\sum xy = 9$, and $\sum x^2 = 5$. Then

$$
b_1 = \dfrac{3\cdot 9 - 3\cdot 6}{3\cdot 5 - 9} = \dfrac{9}{6} = \dfrac{3}{2}, \qquad
b_0 = \dfrac{6 - \tfrac{3}{2}\cdot 3}{3} = \dfrac{1}{2}.
$$

The prediction at $x = 2$ is $1/2 + 3 = 7/2 = 3.5$, whereas the weighted prediction was $27/7 \approx 3.86$, closer to the trusted response $4$. Changing only the weights changed the line.


In [2]:
# Equal weights pull the prediction at x=2 away from the trusted response.
from fractions import Fraction
intercept, slope = Fraction(1, 2), Fraction(3, 2)
ordinary_at_2 = intercept + slope * 2
weighted_at_2 = Fraction(27, 7)
print("ordinary prediction", ordinary_at_2)
print("weighted prediction", weighted_at_2)
assert ordinary_at_2 == Fraction(7, 2)
assert abs(weighted_at_2 - 4) < abs(ordinary_at_2 - 4)


ordinary prediction 7/2
weighted prediction 27/7


## Pitfall

A weight is not a prize for a small residual. Choosing $w_i$ after seeing $e_i$, and giving large weights to rows the first fit already matched, moves the fit toward those rows and then makes the weighted residual sum of squares look artificially small. In the derivation, $w_i$ is part of the model, proportional to the precision, and it is fixed before the residuals are computed. Here the weight $4$ was attached to $x = 2$ before either line was fitted.


In [3]:
# The weights were declared with the data, not computed from the residuals.
weights = [1, 1, 4]
print("weights", weights)
assert weights == [1, 1, 4]
assert all(weight > 0 for weight in weights)


weights [1, 1, 4]


## Summary

- Weighted least squares solves $X^{\mathsf T}W X\,\beta = X^{\mathsf T}W y$.
- With weights $1, 1, 4$, the line is $\tfrac{3}{7} + \tfrac{12}{7}x$ and the weighted cost is $12/7$.
- Equal weights on the same points give $\tfrac{1}{2} + \tfrac{3}{2}x$. The weights are part of the model, not a function of the residuals.
